<img src="images/dask-logo.svg" width="20%" align="right"/>

# Big data analysis with Dask

In this notebook, we'll work some specific computations and learn some best practices for distributed computing.

---

_Note: There are some "Wall time ..." comments added to the cells, these will change depending on your cluster/profiles, etc. So use them only as a rough reference._


## Connect to your Dask Cluster

Instead of starting a new cluster each time, you can view and connect to existing clusters.

In [7]:
# Cài đặt Dask Local Client, hvplot và các phụ thuộc
!pip install -q "dask[complete]" hvplot

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 18.1 MB/s eta 0:00:00


In [2]:
!pip install -q "dask[distributed]" hvplot

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.6/180.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 1.8 MB/s eta 0:00:00


In [3]:
from dask.distributed import Client, LocalCluster

cluster = LocalCluster()
client = Client(cluster)

client

INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:37987
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:43753'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:45027'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:44537 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:44537
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:42932
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:35389 name: 1
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:35389
INFO:distributed.core:Starting established connection to tcp://127

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 2
Total threads: 2,Total memory: 12.67 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:37987,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:44537,Total threads: 1
Dashboard: http://127.0.0.1:43631/status,Memory: 6.34 GiB
Nanny: tcp://127.0.0.1:43753,


In [4]:
from dask.distributed import Client, LocalCluster

# Nếu chưa có cluster thì tạo mới
if "client" not in globals():
    cluster = LocalCluster()
    client = Client(cluster)

client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 2
Total threads: 2,Total memory: 12.67 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:37987,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:44537,Total threads: 1
Dashboard: http://127.0.0.1:43631/status,Memory: 6.34 GiB
Nanny: tcp://127.0.0.1:43753,


List your previous created clusters:

In [5]:
print("Đang dùng LocalCluster trên Google Colab")

print("Cluster:")
print(cluster)

print("\nClient:")
print(client)

Đang dùng LocalCluster trên Google Colab
Cluster:
LocalCluster(7a5f34a3, 'tcp://127.0.0.1:37987', workers=2, threads=2, memory=12.67 GiB)

Client:
<Client: 'tcp://127.0.0.1:37987' processes=2 threads=2, memory=12.67 GiB>


Connect to the cluster with it's name, the previously selected options will be preserved:

In [6]:
cluster = gateway.connect(running_clusters[0].name)

NameError: name 'gateway' is not defined

In [ ]:
# # If there aren't any active clusters

# options = gateway.cluster_options(use_local_defaults=False)
# options.profile = "Medium Worker"
# options.conda_environment = "scipy/scipy-data-of-unusual-size"

# cluster = gateway.new_cluster(options)
# cluster.adapt(minimum=3, maximum=10)

In [ ]:
cluster

Finally, connect a client to the cluster -- so that you can access it from this specific IPython notebook.

In [ ]:
client = cluster.get_client()
client

Open the Dashboard plots: Cluster Map, Progress, Task Stream, and Workers Memory.

## Read the full dataset

We'll work with the Parquet datasets moving forward!

In [ ]:
import dask.dataframe as dd

In [ ]:
import dask.dataframe as dd
import requests
import zipfile
import io
import os

# Link dữ liệu tháng 1/2022
url = (
    "https://transtats.bts.gov/PREZIP/"
    "On_Time_Reporting_Carrier_On_Time_Performance_"
    "1987_present_2022_1.zip"
)

# Tải file ZIP
r = requests.get(url)
r.raise_for_status()

# Giải nén CSV vào /content
with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    csv_file = [f for f in z.namelist() if f.endswith(".csv")][0]
    z.extract(csv_file, "/content")

csv_path = os.path.join("/content", csv_file)

print("Đã tải CSV:", csv_path)

# Khai báo kiểu dữ liệu để tránh lỗi Dask
dtypes = {
    "ArrTime": "float64",
    "ArrivalDelayGroups": "float64",
    "CancellationCode": "object",
    "DepTime": "float64",
    "DepartureDelayGroups": "float64",
    "Div1Airport": "object",
    "Div1TailNum": "object",
    "Div2Airport": "object",
    "Div2TailNum": "object",
    "WheelsOff": "float64",
    "WheelsOn": "float64"
}

# Đọc CSV bằng Dask
ddf_csv = dd.read_csv(
    csv_path,
    dtype=dtypes,
    assume_missing=True
)

# Tạo Parquet
parquet_path = "/content/ontime_2022.parquet"

ddf_csv.to_parquet(
    parquet_path,
    write_index=False
)

print("Đã tạo Parquet:", parquet_path)

# Đọc lại Parquet
ddf = dd.read_parquet(parquet_path)

# Xem 5 dòng đầu
ddf.head()

In [ ]:
ddf.head()

## Shuffling computations

### Calculate and plot the number of canceled flights each day

Note that the following section will take 5+ minutes to compute.

In [ ]:
import hvplot
import hvplot.dask

In [ ]:
hvplot.extension('bokeh') # ensure this cell executes before moving on

In [ ]:
%%time

ddf.groupby("FlightDate")["Cancelled"].count().hvplot()

This takes:

1. so long to compute,
2. the plot lines look out of place,
3. there is so much "red", i.e., interaction, in the task stream

because there's an internal "sort" operation being done.

The "partition" based workflow in Dask makes certain types of operations very expensive. Operations like sort, set_index, etc., require dataset-wide interactions, called "shuffling", making them very difficult to parallelize.

It's a good practice to always **minimize shuffling in distributed computations**.

The Dask dashboard is very helpful in catching these unexpected performance penalties

Red is a universal color in the Dask dashboard, and it always indicated networking costs when there is data transfer between workers. If you hover on the red bars, you'll see they start with `transfer-` or equivalent.

## Read the sorted dataset

If your computes needs some form of sorting or changing index, you should consider doing it once at the beginning and storing/using the sorted dataset.

We have sorted and stored the dataset in the same GCP bucket :)

In [ ]:
import dask.dataframe as dd

ddf = dd.read_parquet(
    "/content/ontime_2022.parquet"
)

ddf.head()

### 💻 Your turn: Run the previous computation on the sorted dataset and compare them

In [ ]:
# Your code here. When ready, click on the three dots below for the solution.
%%time

ddf.groupby("FlightDate")["Cancelled"].count().hvplot()

In [ ]:
%%time

ddf.groupby("FL_DATE")["CANCELLED"].count().hvplot()

## Persisting data and intermediates

A "compute" in Dask creates a pandas/numeric output and bring the final result to your client machine. This is not required in cases where:

- Your client machine (where the output will be displayed) doesn't have enough resources to store/display the output, or
- You have more computations to do with the data or intermediate results, and keeping them on the workers will optimize your overall workflow

### Data locality

Dask tries to assign computations following "data locality", where the computation goes to the worker that holds the required data.

If you remember, data transfer is one of the slowest parts of a workflow.

### 💻 Your turn: Compute the number of departure and arrival delays per day (without persisting)

This is partially similar to the previous workflow where you need to groupby `FL_DATE`. Make sure to record how long this takes!

In [ ]:
# Your code here. When ready, click on the three dots below for the solutions.
ddf_p = ddf[
    ["FlightDate", "DepDelay", "ArrDelay"]
].persist()

In [ ]:
import dask.dataframe as dd
from dask.distributed import Client, LocalCluster, wait

# Đóng cluster cũ nếu còn
try:
    client.close()
    cluster.close()
except:
    pass

# Tạo cluster nhẹ và ổn định hơn cho Google Colab
cluster = LocalCluster(
    n_workers=1,
    threads_per_worker=2,
    processes=False,
    dashboard_address=None
)

client = Client(cluster)

# Đọc lại dữ liệu Parquet
ddf = dd.read_parquet(
    "/content/ontime_2022.parquet"
)

# Chỉ giữ 3 cột cần dùng rồi persist
ddf_p = ddf[
    ["FlightDate", "DepDelay", "ArrDelay"]
].persist()

# Chờ persist hoàn tất
wait(ddf_p)

print("Persist dữ liệu thành công")

In [ ]:
%%time

ddf.groupby("FlightDate")["ArrDelay"].count().compute()

### Persist

Dask allows you to "persist" your data or intermediate outputs (as Dask objects) on the workers.

This can run in the background and the control is returned to you immediately.

### Run the same computation with persisting:

In [ ]:
ddf_p = ddf[
    ["FlightDate", "DepDelay", "ArrDelay"]
].persist()

ddf_p

Open the "Graph" dashboard plot for the following computations!

In [ ]:
ddf_p

Notice how you have the control back immediately, and you can continue working with a Dask DataFrame!

In [ ]:
from dask.distributed import Client, LocalCluster, wait
import dask.dataframe as dd

# Đóng cluster/client cũ nếu còn
try:
    client.close()
except:
    pass

try:
    cluster.close()
except:
    pass

# Tạo LocalCluster ổn định hơn cho Colab
cluster = LocalCluster(
    n_workers=1,
    threads_per_worker=2,
    processes=False,
    dashboard_address=None
)

client = Client(cluster)

# Đọc lại dữ liệu từ Parquet
ddf = dd.read_parquet(
    "/content/ontime_2022.parquet"
)

# Chỉ lấy 3 cột cần cho phần này
ddf_p = ddf[
    ["FlightDate", "DepDelay", "ArrDelay"]
].persist()

# Chờ persist hoàn tất
wait(ddf_p)

print("Persist thành công")
print(ddf_p)

## Partitioning effectively

Our dataset currently has 1251 partitions:

In [ ]:
ddf.npartitions

You can change the number of partitions with: `ddf.repartition(npartitions=xx)`

### 💻 Your turn: Compute the unique flights taken each day, comparing the performance with current partitions and ~600 partitions

In [ ]:
# Your code here. When ready, click on the three dots below for the solutions.
import dask
import dask.dataframe as dd

# Dùng scheduler threads để ổn định trên Google Colab
dask.config.set(scheduler="threads")

# Đọc lại dữ liệu Parquet từ đầu
ddf = dd.read_parquet("/content/ontime_2022.parquet")

# Chỉ lấy 2 cột cần thiết
ddf_full = ddf[
    ["FlightDate", "Reporting_Airline"]
]

# ==========================
# 1. Partition hiện tại
# ==========================

print("Số partition hiện tại:", ddf_full.npartitions)

result_current = (
    ddf_full
    .groupby("FlightDate")
    ["Reporting_Airline"]
    .count()
    .compute()
)

print("\nKết quả với partition hiện tại:")
print(result_current.head())


# ==========================
# 2. Repartition
# ==========================

ddf_600 = ddf_full.repartition(
    npartitions=600
)

print("\nSố partition sau repartition:", ddf_600.npartitions)

result_600 = (
    ddf_600
    .groupby("FlightDate")
    ["Reporting_Airline"]
    .count()
    .compute()
)

print("\nKết quả với 600 partitions:")
print(result_600.head())

In [ ]:
ddf_full = ddf[
    ["FlightDate", "Reporting_Airline"]
]

In [ ]:
%%time

ddf_full.groupby(
    "FlightDate"
)["Reporting_Airline"].count().compute()

In [ ]:
ddf_600 = ddf_full.repartition(npartitions=600).persist()

In [ ]:
%%time

ddf_600.groupby(
    "FlightDate"
)["Reporting_Airline"].count().compute()

Re-partitioning is an expensive operation. Notice all the red bars in the task graph.

Therefore, when you store your data, ensure you have the optimal number of partitions depending on your dataset, your computation, number of workers, and worker resources.

## `meta` keyword

Since Dask evaluates computations lazily, it uses a special `meta` property to keeps track of the output structure of any computation.

### 💻 Your turn: Convert all negative values in DEP_DELAY to zeros

In [ ]:
# Your code here. When ready, click on the three dots below for the solutions.
f = lambda x: 0 if x < 0 else x

ddf["DepDelay"].apply(f).compute()

In [ ]:
f = lambda x: 0 if x < 0 else x

ddf["DepDelay"].apply(f).compute()

This is a common warning. `f` is a custom function to Dask can not reliable predict the output structure. In such cases, it's best to specify `meta` explicitly.

### Specify `meta`:

You can specify `meta` with an empty/sample pandas data structure with the appropriate columns names and data types.

Optional, further reading: [Understanding Dask’s meta keyword argument](https://blog.dask.org/2022/08/09/understanding-meta-keyword-argument)

In [ ]:
import pandas as pd

In [ ]:
import pandas as pd

f = lambda x: 0 if x < 0 else x

ddf["DepDelay"].apply(
    f,
    meta=pd.Series(dtype="float64")
).compute()

---

## Next →

Let's look at [big data visualizations](./06-big-data-visualization.ipynb)!